## 0. Obiettivo del notebook

Questo notebook è costruito per produrre in modo riproducibile:

- i grafici, le tabelle e le analisi del **Capitolo 3**;
- la validazione statistica aggiuntiva con **k-fold**;
- la sezione di **interpretabilità con SHAP**;
- la validazione esterna su **CIC-IoT-Dataset2023** per il confronto cross-dataset.


In [1]:
from __future__ import annotations

import time
import tempfile
from pathlib import Path
from typing import List, Tuple, Dict, Any

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from IPython.display import display

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
    precision_recall_curve,
    auc,
)
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, label_binarize, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

HAS_XGB = True
HAS_LGBM = True
HAS_SHAP = True

try:
    import shap
except Exception as exc:
    HAS_SHAP = False
    shap = None
    print(f"[WARN] shap non disponibile: {exc}")

try:
    from xgboost import XGBClassifier
except Exception as exc:
    HAS_XGB = False
    XGBClassifier = None
    print(f"[WARN] xgboost non disponibile: {exc}")

try:
    from lightgbm import LGBMClassifier
except Exception as exc:
    HAS_LGBM = False
    LGBMClassifier = None
    print(f"[WARN] lightgbm non disponibile: {exc}")


In [2]:
# =========================
# CONFIGURAZIONE GENERALE
# =========================
RANDOM_STATE = 42
TEST_SIZE = 0.20
TOP_MISSING = 20
HIGH_CARDINALITY_THRESHOLD = 50

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data").exists() and (PROJECT_ROOT.parent / "data").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_PATH = PROJECT_ROOT / "data" / "train_test_network.csv"
CIC_DATA_DIR = PROJECT_ROOT / "data" / "CIC-IoT-Dataset2023"

FIGURES_DIR = PROJECT_ROOT / "figures"
TABLES_DIR = PROJECT_ROOT / "tables"
MODELS_DIR = PROJECT_ROOT / "models"
OUTPUT_DIR = PROJECT_ROOT / "outputs"

for folder in [FIGURES_DIR, TABLES_DIR, MODELS_DIR, OUTPUT_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

sns.set_theme(style="whitegrid")
plt.rcParams.update({
    "figure.figsize": (10, 6),
    "axes.titlesize": 16,
    "axes.labelsize": 13,
    "xtick.labelsize": 11,
    "ytick.labelsize": 11,
    "legend.fontsize": 11,
    "savefig.dpi": 300,
})

print(f"Project root: {PROJECT_ROOT}")
print(f"Dataset path: {DATA_PATH}")
print(f"CIC-IoT directory: {CIC_DATA_DIR}")


Project root: C:\Users\emanu\OneDrive\Desktop\iot-audit
Dataset path: C:\Users\emanu\OneDrive\Desktop\iot-audit\data\train_test_network.csv
CIC-IoT directory: C:\Users\emanu\OneDrive\Desktop\iot-audit\data\CIC-IoT-Dataset2023


## 3.1 Caricamento e controllo iniziale del dataset


In [3]:
if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Dataset non trovato in: {DATA_PATH}. "
        "Verifica di aver copiato train_test_network.csv nella cartella data/."
    )

df_raw = pd.read_csv(DATA_PATH)
print("Shape dataset:", df_raw.shape)
display(df_raw.head(3))

Shape dataset: (211043, 44)


,src_ip,src_port,dst_ip,dst_port,proto,service,duration,src_bytes,dst_bytes,conn_state,...,http_response_body_len,http_status_code,http_user_agent,http_orig_mime_types,http_resp_mime_types,weird_name,weird_addl,weird_notice,label,type
0,192.168.1.37,4444,192.168.1.193,49178,tcp,-,290.371539,101568,2592,OTH,...,0,0,-,-,-,-,-,-,1,backdoor
1,192.168.1.193,49180,192.168.1.37,8080,tcp,-,0.000102,0,0,REJ,...,0,0,-,-,-,-,-,-,1,backdoor
2,192.168.1.193,49180,192.168.1.37,8080,tcp,-,0.000148,0,0,REJ,...,0,0,-,-,-,-,-,-,1,backdoor


In [4]:
print("Colonne disponibili:")
print(df_raw.columns.tolist())

print("\nTipi di dato:")
display(df_raw.dtypes.value_counts())

print("\nInformazioni sintetiche:")
df_raw.info()

Colonne disponibili:
['src_ip', 'src_port', 'dst_ip', 'dst_port', 'proto', 'service', 'duration', 'src_bytes', 'dst_bytes', 'conn_state', 'missed_bytes', 'src_pkts', 'src_ip_bytes', 'dst_pkts', 'dst_ip_bytes', 'dns_query', 'dns_qclass', 'dns_qtype', 'dns_rcode', 'dns_AA', 'dns_RD', 'dns_RA', 'dns_rejected', 'ssl_version', 'ssl_cipher', 'ssl_resumed', 'ssl_established', 'ssl_subject', 'ssl_issuer', 'http_trans_depth', 'http_method', 'http_uri', 'http_version', 'http_request_body_len', 'http_response_body_len', 'http_status_code', 'http_user_agent', 'http_orig_mime_types', 'http_resp_mime_types', 'weird_name', 'weird_addl', 'weird_notice', 'label', 'type']

Tipi di dato:


str        27
int64      16
float64     1
Name: count, dtype: int64


Informazioni sintetiche:
<class 'pandas.DataFrame'>
RangeIndex: 211043 entries, 0 to 211042
Data columns (total 44 columns):
 #   Column                  Non-Null Count   Dtype  
---  ------                  --------------   -----  
 0   src_ip                  211043 non-null  str    
 1   src_port                211043 non-null  int64  
 2   dst_ip                  211043 non-null  str    
 3   dst_port                211043 non-null  int64  
 4   proto                   211043 non-null  str    
 5   service                 211043 non-null  str    
 6   duration                211043 non-null  float64
 7   src_bytes               211043 non-null  int64  
 8   dst_bytes               211043 non-null  int64  
 9   conn_state              211043 non-null  str    
 10  missed_bytes            211043 non-null  int64  
 11  src_pkts                211043 non-null  int64  
 12  src_ip_bytes            211043 non-null  int64  
 13  dst_pkts                211043 non-null  int64  
 14  dst_i

In [5]:
summary_quality = pd.DataFrame({
    "missing_values": df_raw.isna().sum(),
    "missing_pct": (df_raw.isna().sum() / len(df_raw) * 100).round(2),
    "n_unique": df_raw.nunique(dropna=True),
}).sort_values("missing_values", ascending=False)

display(summary_quality.head(20))
print(f"Duplicati: {df_raw.duplicated().sum()}")

,missing_values,missing_pct,n_unique
src_ip,0,0.0,51
src_port,0,0.0,26628
dst_ip,0,0.0,753
dst_port,0,0.0,2039
proto,0,0.0,3
service,0,0.0,9
duration,0,0.0,68570
src_bytes,0,0.0,2199
dst_bytes,0,0.0,2338
conn_state,0,0.0,13


Duplicati: 20569


## 3.2 Preparazione di base e pulizia controllata



In [6]:
TARGET_BINARY = "label"
TARGET_MULTICLASS = "type"

manual_drop_candidates = [
    "src_ip", "dst_ip",
    "http_uri", "ssl_subject", "ssl_issuer",
    "dns_query", "http_host", "user_agent",
]

def prepare_base_dataframe(df: pd.DataFrame) -> Tuple[pd.DataFrame, List[str]]:
    df = df.copy()

    # Normalizza placeholder testuali comuni
    df = df.replace(["-", "NA", "N/A", ""], np.nan)

    # Selezione colonne da eliminare: manuali + alta cardinalità testuale
    drop_cols = [c for c in manual_drop_candidates if c in df.columns]

    text_cols = df.select_dtypes(include=["object", "string"]).columns.tolist()
    text_cols = [c for c in text_cols if c not in [TARGET_BINARY, TARGET_MULTICLASS]]

    for col in text_cols:
        nunique = df[col].nunique(dropna=True)
        if nunique > HIGH_CARDINALITY_THRESHOLD and col not in drop_cols:
            drop_cols.append(col)

    # Colonne costanti
    constant_cols = [
        c for c in df.columns
        if c not in [TARGET_BINARY, TARGET_MULTICLASS] and df[c].nunique(dropna=True) <= 1
    ]
    drop_cols.extend([c for c in constant_cols if c not in drop_cols])

    # Elimina duplicati
    df = df.drop_duplicates()

    # Drop finale
    existing_drop_cols = [c for c in drop_cols if c in df.columns]
    df = df.drop(columns=existing_drop_cols, errors="ignore")

    return df, existing_drop_cols

df, dropped_columns = prepare_base_dataframe(df_raw)

print("Colonne eliminate:", dropped_columns)
print("Shape dopo pulizia:", df.shape)
display(df.head(3))

Colonne eliminate: ['src_ip', 'dst_ip', 'http_uri', 'ssl_subject', 'ssl_issuer', 'dns_query', 'http_version', 'weird_notice']
Shape dopo pulizia: (190474, 36)


,src_port,dst_port,proto,service,duration,src_bytes,dst_bytes,conn_state,missed_bytes,src_pkts,...,http_request_body_len,http_response_body_len,http_status_code,http_user_agent,http_orig_mime_types,http_resp_mime_types,weird_name,weird_addl,label,type
0,4444,49178,tcp,NaN,290.371539,101568,2592,OTH,0,108,...,0,0,0,NaN,NaN,NaN,NaN,NaN,1,backdoor
1,49180,8080,tcp,NaN,0.000102,0,0,REJ,0,1,...,0,0,0,NaN,NaN,NaN,NaN,NaN,1,backdoor
2,49180,8080,tcp,NaN,0.000148,0,0,REJ,0,1,...,0,0,0,NaN,NaN,NaN,NaN,NaN,1,backdoor


## 3.3 Grafici e analisi esplorativa per il Capitolo 3


In [7]:
def save_figure(path: Path) -> None:
    plt.tight_layout()
    plt.savefig(path, dpi=300, bbox_inches="tight")
    plt.close()

def annotate_bars(ax):
    for p in ax.patches:
        height = p.get_height()
        ax.annotate(
            f"{int(height):,}".replace(",", "."),
            (p.get_x() + p.get_width() / 2, height),
            ha="center",
            va="bottom",
            fontsize=9,
            xytext=(0, 3),
            textcoords="offset points",
        )

def plot_distribution(series: pd.Series, title: str, xlabel: str, filename: str, rotate=0):
    counts = series.value_counts(dropna=False)
    fig, ax = plt.subplots()
    ax.bar(counts.index.astype(str), counts.values)
    ax.set_title(title)
    ax.set_xlabel(xlabel)
    ax.set_ylabel("Numero di flussi")
    ax.tick_params(axis="x", rotation=rotate)
    annotate_bars(ax)
    save_figure(FIGURES_DIR / filename)
    plt.show()
    return counts

binary_counts = plot_distribution(
    df[TARGET_BINARY],
    "Distribuzione delle classi - Task binario",
    "Classe",
    "class_distribution_binary.png",
    rotate=0
)

multiclass_counts = plot_distribution(
    df[TARGET_MULTICLASS],
    "Distribuzione delle classi - Task multiclass",
    "Tipo di traffico / attacco",
    "class_distribution_multiclass.png",
    rotate=45
)

display(binary_counts.to_frame("count"))
display(multiclass_counts.to_frame("count").head(20))

,count
label,
1,148434
0,42040


,count
type,
normal,42040
scanning,20000
ddos,19993
injection,19964
password,19861
dos,18992
backdoor,18711
xss,15137
ransomware,14735


In [8]:
# Missing values: top 20 colonne
missing = df.isna().sum().sort_values(ascending=False)
missing = missing[missing > 0]

if len(missing) > 0:
    fig, ax = plt.subplots(figsize=(12, 6))
    missing.head(TOP_MISSING).plot(kind="bar", ax=ax)
    ax.set_title("Valori mancanti - Top colonne")
    ax.set_xlabel("Colonna")
    ax.set_ylabel("Numero di valori mancanti")
    ax.tick_params(axis="x", rotation=45)
    save_figure(FIGURES_DIR / "missing_values_top20.png")
    display(missing.head(TOP_MISSING).to_frame("missing_values"))
else:
    print("Nessun valore mancante rilevato.")

,missing_values
http_orig_mime_types,190458
weird_addl,190317
http_resp_mime_types,190270
http_method,190187
http_user_agent,190187
http_trans_depth,190171
weird_name,190118
ssl_version,190073
ssl_established,190073
ssl_resumed,190073


In [9]:
# Heatmap di correlazione sulle variabili numeriche
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()

if len(numeric_cols) >= 2:
    corr = df[numeric_cols].corr()

    plt.figure(figsize=(14, 10))
    sns.heatmap(corr, cmap="coolwarm", center=0, square=False, cbar_kws={"shrink": 0.8})
    plt.title("Matrice di correlazione delle variabili numeriche")
    save_figure(FIGURES_DIR / "correlation_heatmap.png")
else:
    print("Numero insufficiente di colonne numeriche per la heatmap.")

In [10]:
binary_pct = (binary_counts / binary_counts.sum() * 100).round(2)
display(pd.DataFrame({"count": binary_counts, "percentage": binary_pct}))

,count,percentage
label,,
1,148434,77.93
0,42040,22.07


## 3.4 Funzioni di supporto per gli esperimenti


In [11]:
from utils import (
    build_preprocessor,
    get_models,
    evaluate_binary_pipeline,
    evaluate_multiclass_pipeline,
    cross_validate_binary_models,
    measure_inference_time_ms,
    plot_confusion_matrix,
    plot_metric_comparison,
    plot_cv_metric_comparison,
)

## 3.5 Esperimento 1 — Risultati classificazione binaria


In [12]:
binary_df = df.copy()
binary_df = binary_df.drop(columns=[TARGET_MULTICLASS], errors="ignore")

X_binary = binary_df.drop(columns=[TARGET_BINARY])
y_binary = binary_df[TARGET_BINARY]

# sicurezza su tipo label
if y_binary.dtype == object:
    y_binary = y_binary.astype(str)

X_train_b, X_test_b, y_train_b, y_test_b = train_test_split(
    X_binary,
    y_binary,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y_binary
)

# usa utils.py
preprocessor_b, num_feat_b, cat_feat_b = build_preprocessor(X_train_b)

print(f"Feature numeriche: {len(num_feat_b)}")
print(f"Feature categoriche: {len(cat_feat_b)}")
print(f"Train shape: {X_train_b.shape} | Test shape: {X_test_b.shape}")

Feature numeriche: 16
Feature categoriche: 17
Train shape: (152379, 34) | Test shape: (38095, 34)


In [13]:
binary_models = get_models(task="binary")

binary_results = []
binary_predictions = {}
binary_probabilities = {}
binary_pipelines = {}

for model_name, estimator in binary_models.items():
    print(f"Training binary model: {model_name}")
    pipe = Pipeline(steps=[
        ("preprocessor", preprocessor_b),
        ("model", estimator),
    ])
    pipe.fit(X_train_b, y_train_b)

    result, y_pred, y_prob = evaluate_binary_pipeline(pipe, X_test_b, y_test_b, model_name)
    binary_results.append(result)
    binary_predictions[model_name] = y_pred
    binary_probabilities[model_name] = y_prob
    binary_pipelines[model_name] = pipe

binary_results_df = pd.DataFrame(binary_results).sort_values(by="f1", ascending=False)
display(binary_results_df)
binary_results_df.to_csv(OUTPUT_DIR / "binary_results.csv", index=False)

Training binary model: Logistic Regression
Training binary model: Random Forest
Training binary model: XGBoost
Training binary model: LightGBM
[LightGBM] [Info] Number of positive: 118747, number of negative: 33632
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.013357 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2452
[LightGBM] [Info] Number of data points in the train set: 152379, number of used features: 63
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.779287 -> initscore=1.261517
[LightGBM] [Info] Start training from score 1.261517


C:\Users\emanu\OneDrive\Desktop\iot-audit\venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
C:\Users\emanu\OneDrive\Desktop\iot-audit\venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
C:\Users\emanu\OneDrive\Desktop\iot-audit\venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
C:\Users\emanu\OneDrive\Desktop\iot-audit\venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
C:\Users\emanu\OneDrive\Desktop\iot-audit\venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid featur

,model,accuracy,precision,recall,f1,roc_auc,pr_auc,inference_ms_per_1k,model_size_mb
3,LightGBM,0.998819,0.998956,0.999528,0.999242,0.999988,0.999997,38.77367,1.370345
1,Random Forest,0.998504,0.998687,0.999394,0.999040,0.999980,0.999994,62.14583,21.902629
2,XGBoost,0.998320,0.998653,0.999192,0.998922,0.999975,0.999993,17.40695,0.757490
0,Logistic Regression,0.984407,0.990293,0.989692,0.989993,0.994463,0.998091,24.76429,0.010355


In [14]:
plot_metric_comparison(binary_results_df, "f1", "Confronto F1-score (Binary)", FIGURES_DIR / "binary_f1_comparison.png")
plot_metric_comparison(binary_results_df, "roc_auc", "Confronto ROC-AUC (Binary)", FIGURES_DIR / "binary_roc_auc_comparison.png")
plot_metric_comparison(binary_results_df, "pr_auc", "Confronto PR-AUC (Binary)", FIGURES_DIR / "binary_pr_auc_comparison.png")
plot_metric_comparison(binary_results_df, "inference_ms_per_1k", "Confronto latenza (ms / 1000 predizioni) - Binary", FIGURES_DIR / "binary_latency_comparison.png", sort_desc=False)
plot_metric_comparison(binary_results_df, "model_size_mb", "Confronto dimensione modello (MB) - Binary", FIGURES_DIR / "binary_size_comparison.png", sort_desc=False)

In [15]:
best_binary = binary_results_df.iloc[0]["model"]
print("Miglior modello binary:", best_binary)

plot_confusion_matrix(
    y_test_b,
    binary_predictions[best_binary],
    labels=sorted(pd.unique(y_binary).tolist()),
    title=f"Confusion Matrix Binary - {best_binary}",
    filename=FIGURES_DIR / "binary_confusion_matrix_best.png",
    normalize=False,
)

best_prob = binary_probabilities[best_binary]
fpr, tpr, _ = roc_curve(y_test_b, best_prob)
precision, recall, _ = precision_recall_curve(y_test_b, best_prob)
pr_auc_value = auc(recall, precision)

fig, ax = plt.subplots(figsize=(8, 6))
ax.plot(fpr, tpr, label=f"{best_binary}")
ax.plot([0, 1], [0, 1], linestyle="--")
ax.set_title(f"ROC Curve Binary - {best_binary}")
ax.set_xlabel("False Positive Rate")
ax.set_ylabel("True Positive Rate")
ax.legend()
save_figure(FIGURES_DIR / "binary_roc_curve_best.png")

fig, ax = plt.subplots(figsize=(8, 6))
ax.plot(recall, precision, label=f"{best_binary} (AUC={pr_auc_value:.3f})")
ax.set_title(f"Precision-Recall Curve Binary - {best_binary}")
ax.set_xlabel("Recall")
ax.set_ylabel("Precision")
ax.legend()
save_figure(FIGURES_DIR / "binary_pr_curve_best.png")

Miglior modello binary: LightGBM


## 3.6 Esperimento 2 — Risultati classificazione multiclass


In [16]:
from sklearn.preprocessing import LabelEncoder

multiclass_df = df.copy()
multiclass_df = multiclass_df.drop(columns=[TARGET_BINARY], errors="ignore")

X_multi = multiclass_df.drop(columns=[TARGET_MULTICLASS])
y_multi = multiclass_df[TARGET_MULTICLASS].astype(str)


from sklearn.preprocessing import LabelEncoder

label_encoder = LabelEncoder()
y_multi_encoded = label_encoder.fit_transform(y_multi)


X_train_m, X_test_m, y_train_m, y_test_m = train_test_split(
    X_multi,
    y_multi_encoded,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y_multi_encoded
)

preprocessor_m, numeric_features_m, categorical_features_m = build_preprocessor(X_train_m)

print("Feature numeriche:", len(numeric_features_m))
print("Feature categoriche:", len(categorical_features_m))
print("Classi multiclass:", len(label_encoder.classes_))
print("Train shape:", X_train_m.shape, "Test shape:", X_test_m.shape)

Feature numeriche: 16
Feature categoriche: 17
Classi multiclass: 10
Train shape: (152379, 34) Test shape: (38095, 34)


In [17]:
multiclass_models = get_models(
    task="multiclass",
    n_classes=len(label_encoder.classes_)
)

multiclass_results = []
multiclass_predictions = {}
multiclass_probabilities = {}
multiclass_pipelines = {}
multiclass_labels = None

for model_name, estimator in multiclass_models.items():
    print(f"Training multiclass model: {model_name}")
    
    pipe = Pipeline(steps=[
        ("preprocessor", preprocessor_m),
        ("model", estimator),
    ])
    
    pipe.fit(X_train_m, y_train_m)

    result, y_pred, y_prob, labels = evaluate_multiclass_pipeline(
        pipe, X_test_m, y_test_m, model_name
    )

    y_pred_labels = label_encoder.inverse_transform(y_pred)

    multiclass_results.append(result)
    multiclass_predictions[model_name] = y_pred_labels
    multiclass_probabilities[model_name] = y_prob
    multiclass_pipelines[model_name] = pipe
    multiclass_labels = labels

Training multiclass model: Logistic Regression
Training multiclass model: Random Forest
Training multiclass model: XGBoost
Training multiclass model: LightGBM
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.024932 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 2450
[LightGBM] [Info] Number of data points in the train set: 152379, number of used features: 62
[LightGBM] [Info] Start training from score -2.320389
[LightGBM] [Info] Start training from score -2.254157
[LightGBM] [Info] Start training from score -2.305536
[LightGBM] [Info] Start training from score -2.255596
[LightGBM] [Info] Start training from score -5.209092
[LightGBM] [Info] Start training from score -1.510893
[LightGBM] [Info] Start training from score -2.260744
[LightGBM] [Info] Start training from score -2.559289
[LightGBM] [Info] Start training from score -2.253782
[LightGBM] [Info] Start training from score -2.532339
[LightGBM] 

C:\Users\emanu\OneDrive\Desktop\iot-audit\venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
C:\Users\emanu\OneDrive\Desktop\iot-audit\venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
C:\Users\emanu\OneDrive\Desktop\iot-audit\venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
C:\Users\emanu\OneDrive\Desktop\iot-audit\venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
C:\Users\emanu\OneDrive\Desktop\iot-audit\venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid featur

In [18]:
multiclass_results_df = pd.DataFrame(multiclass_results).sort_values(by="macro_f1", ascending=False)
display(multiclass_results_df)
multiclass_results_df.to_csv(OUTPUT_DIR / "multiclass_results.csv", index=False)

,model,accuracy,macro_f1,weighted_f1,macro_precision,macro_recall,roc_auc_macro,pr_auc_macro,inference_ms_per_1k,model_size_mb
2,XGBoost,0.989605,0.969444,0.989646,0.966955,0.972163,0.999801,0.985043,29.03297,7.192165
3,LightGBM,0.989684,0.969250,0.989674,0.970045,0.968505,0.999829,0.984993,129.44168,13.133498
1,Random Forest,0.987977,0.967793,0.988033,0.964593,0.971315,0.999673,0.986230,104.62015,180.436168
0,Logistic Regression,0.852605,0.790323,0.849685,0.810586,0.785785,0.985411,0.835826,8.17224,0.016878


In [19]:
plot_metric_comparison(multiclass_results_df, "accuracy", "Confronto Accuracy (Multiclass)", FIGURES_DIR / "multiclass_accuracy_comparison.png")
plot_metric_comparison(multiclass_results_df, "macro_f1", "Confronto Macro-F1 (Multiclass)", FIGURES_DIR / "multiclass_macro_f1_comparison.png")
plot_metric_comparison(multiclass_results_df, "weighted_f1", "Confronto Weighted F1 (Multiclass)", FIGURES_DIR / "multiclass_weighted_f1_comparison.png")
plot_metric_comparison(multiclass_results_df, "roc_auc_macro", "Confronto ROC-AUC Macro (Multiclass)", FIGURES_DIR / "multiclass_roc_auc_macro_comparison.png")
plot_metric_comparison(multiclass_results_df, "pr_auc_macro", "Confronto PR-AUC Macro (Multiclass)", FIGURES_DIR / "multiclass_pr_auc_macro_comparison.png")
plot_metric_comparison(multiclass_results_df, "inference_ms_per_1k", "Confronto latenza (ms / 1000 predizioni) - Multiclass", FIGURES_DIR / "multiclass_latency_comparison.png", sort_desc=False)
plot_metric_comparison(multiclass_results_df, "model_size_mb", "Confronto dimensione modello (MB) - Multiclass", FIGURES_DIR / "multiclass_size_comparison.png", sort_desc=False)

In [20]:
best_multi = multiclass_results_df.iloc[0]["model"]
print("Miglior modello multiclass:", best_multi)

y_test_labels = label_encoder.inverse_transform(y_test_m)

y_pred_labels = multiclass_predictions[best_multi]

plot_confusion_matrix(
    y_test_labels,
    y_pred_labels,
    labels=label_encoder.classes_,
    title=f"Confusion Matrix Multiclass - {best_multi}",
    filename=FIGURES_DIR / "multiclass_confusion_matrix_best.png",
    normalize=False,
)

report = classification_report(
    y_test_labels,
    y_pred_labels,
    output_dict=True,
    zero_division=0,
)

report_df = pd.DataFrame(report).T
display(report_df)
report_df.to_csv(OUTPUT_DIR / "multiclass_classification_report_best.csv")

Miglior modello multiclass: XGBoost


,precision,recall,f1-score,support
backdoor,1.000000,1.000000,1.000000,3742.000000
ddos,0.990433,0.983746,0.987078,3999.000000
dos,0.993356,0.983943,0.988627,3799.000000
injection,0.984729,0.968946,0.976774,3993.000000
mitm,0.761261,0.812500,0.786047,208.000000
normal,0.996674,0.997859,0.997266,8408.000000
password,0.996456,0.990937,0.993688,3972.000000
ransomware,0.998983,1.000000,0.999491,2947.000000
scanning,0.986875,0.996250,0.991540,4000.000000
xss,0.960784,0.987446,0.973933,3027.000000


In [21]:
per_class_f1 = report_df.loc[
    [c for c in report_df.index if c not in ["accuracy", "macro avg", "weighted avg"]],
    "f1-score"
].sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(12, 6))
ax.bar(per_class_f1.index, per_class_f1.values)
ax.set_title(f"F1-score per classe - {best_multi}")
ax.set_xlabel("Classe")
ax.set_ylabel("F1-score")
ax.tick_params(axis="x", rotation=45)
annotate_bars(ax)
save_figure(FIGURES_DIR / "multiclass_per_class_f1_best.png")

## 3.8 Validazione statistica e robustezza del modello (k-fold)

La validazione incrociata viene applicata al task binario principale per stimare la stabilità delle prestazioni al variare della partizione dei dati.


In [22]:
# =========================
# 3.8 VALIDAZIONE k-fold - TASK BINARIO
# =========================
binary_cv_models = get_models(task="binary")

binary_cv_folds_df, binary_cv_summary_df = cross_validate_binary_models(
    X_binary,
    y_binary,
    task_name="TON_IoT binary",
    n_splits=5,
    models=binary_cv_models,
)

display(binary_cv_summary_df)
binary_cv_summary_df.to_csv(OUTPUT_DIR / "ton_binary_cv_summary.csv", index=False)
binary_cv_folds_df.to_csv(OUTPUT_DIR / "ton_binary_cv_folds.csv", index=False)

plot_cv_metric_comparison(
    binary_cv_summary_df,
    "f1",
    "Cross-validation 5-fold - F1-score (Binary, TON_IoT)",
    FIGURES_DIR / "ton_binary_cv_f1_comparison.png",
)

plot_cv_metric_comparison(
    binary_cv_summary_df,
    "pr_auc",
    "Cross-validation 5-fold - PR-AUC (Binary, TON_IoT)",
    FIGURES_DIR / "ton_binary_cv_pr_auc_comparison.png",
)

plot_cv_metric_comparison(
    binary_cv_summary_df,
    "roc_auc",
    "Cross-validation 5-fold - ROC-AUC (Binary, TON_IoT)",
    FIGURES_DIR / "ton_binary_cv_roc_auc_comparison.png",
)


[LightGBM] [Info] Number of positive: 118747, number of negative: 33632
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.022868 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2442
[LightGBM] [Info] Number of data points in the train set: 152379, number of used features: 62
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.779287 -> initscore=1.261517
[LightGBM] [Info] Start training from score 1.261517


C:\Users\emanu\OneDrive\Desktop\iot-audit\venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
C:\Users\emanu\OneDrive\Desktop\iot-audit\venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
C:\Users\emanu\OneDrive\Desktop\iot-audit\venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
C:\Users\emanu\OneDrive\Desktop\iot-audit\venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
C:\Users\emanu\OneDrive\Desktop\iot-audit\venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid featur

[LightGBM] [Info] Number of positive: 118747, number of negative: 33632
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.011256 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2439
[LightGBM] [Info] Number of data points in the train set: 152379, number of used features: 63
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.779287 -> initscore=1.261517
[LightGBM] [Info] Start training from score 1.261517


C:\Users\emanu\OneDrive\Desktop\iot-audit\venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
C:\Users\emanu\OneDrive\Desktop\iot-audit\venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
C:\Users\emanu\OneDrive\Desktop\iot-audit\venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
C:\Users\emanu\OneDrive\Desktop\iot-audit\venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
C:\Users\emanu\OneDrive\Desktop\iot-audit\venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid featur

[LightGBM] [Info] Number of positive: 118747, number of negative: 33632
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.010401 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2517
[LightGBM] [Info] Number of data points in the train set: 152379, number of used features: 63
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.779287 -> initscore=1.261517
[LightGBM] [Info] Start training from score 1.261517


...

[LightGBM] [Info] Number of positive: 118747, number of negative: 33632
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.012195 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2456
[LightGBM] [Info] Number of data points in the train set: 152379, number of used features: 62
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.779287 -> initscore=1.261517
[LightGBM] [Info] Start training from score 1.261517


C:\Users\emanu\OneDrive\Desktop\iot-audit\venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
C:\Users\emanu\OneDrive\Desktop\iot-audit\venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
C:\Users\emanu\OneDrive\Desktop\iot-audit\venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
C:\Users\emanu\OneDrive\Desktop\iot-audit\venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
C:\Users\emanu\OneDrive\Desktop\iot-audit\venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid featur

[LightGBM] [Info] Number of positive: 118748, number of negative: 33632
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.011875 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2442
[LightGBM] [Info] Number of data points in the train set: 152380, number of used features: 62
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.779289 -> initscore=1.261526
[LightGBM] [Info] Start training from score 1.261526


C:\Users\emanu\OneDrive\Desktop\iot-audit\venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
C:\Users\emanu\OneDrive\Desktop\iot-audit\venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
C:\Users\emanu\OneDrive\Desktop\iot-audit\venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
C:\Users\emanu\OneDrive\Desktop\iot-audit\venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
C:\Users\emanu\OneDrive\Desktop\iot-audit\venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid featur

,model,task,accuracy_mean,accuracy_std,precision_mean,precision_std,recall_mean,recall_std,f1_mean,f1_std,roc_auc_mean,roc_auc_std,pr_auc_mean,pr_auc_std
3,LightGBM,TON_IoT binary,0.998966,0.000096,0.999178,0.000237,0.999495,0.000130,0.999337,0.000061,0.999984,0.000010,0.999995,0.000003
1,Random Forest,TON_IoT binary,0.998782,0.000186,0.999024,0.000214,0.999414,0.000230,0.999219,0.000119,0.999982,0.000011,0.999995,0.000003
2,XGBoost,TON_IoT binary,0.998603,0.000218,0.998929,0.000249,0.999279,0.000116,0.999104,0.000140,0.999977,0.000012,0.999993,0.000003
0,Logistic Regression,TON_IoT binary,0.985137,0.000556,0.990851,0.000331,0.990070,0.000402,0.990460,0.000357,0.994554,0.000190,0.998211,0.000159


## 3.9 Interpretabilità del modello con SHAP

L'analisi SHAP viene applicata al miglior modello multiclass per spiegare l'importanza globale delle feature e una predizione locale sulla classe MITM.


In [24]:
# =========================
# 3.9 SHAP - MODELLO MULTICLASS MIGLIORE
# =========================
from utils import get_shap_class_index

if HAS_SHAP:
    best_multi_pipeline = multiclass_pipelines[best_multi]
    best_multi_preprocessor = best_multi_pipeline.named_steps["preprocessor"]
    best_multi_model = best_multi_pipeline.named_steps["model"]

    X_multi_trans = best_multi_preprocessor.transform(X_test_m)
    feature_names_multi = best_multi_preprocessor.get_feature_names_out()

    # Campionamento leggero per il summary plot
    shap_sample_n = min(1000, X_multi_trans.shape[0])
    shap_sample_idx = np.random.RandomState(RANDOM_STATE).choice(
        X_multi_trans.shape[0], size=shap_sample_n, replace=False
    )
    X_multi_trans_sample = X_multi_trans[shap_sample_idx]

    explainer = shap.TreeExplainer(best_multi_model)
    shap_values = explainer.shap_values(X_multi_trans_sample)

    mitm_index = get_shap_class_index("mitm", label_encoder)
    if isinstance(shap_values, list):
        shap_values_mitm = shap_values[mitm_index]
        expected_value_mitm = explainer.expected_value[mitm_index]
    else:
        shap_values_mitm = shap_values[:, :, mitm_index] if shap_values.ndim == 3 else shap_values
        expected_value_mitm = explainer.expected_value[mitm_index] if isinstance(explainer.expected_value, (list, np.ndarray)) else explainer.expected_value

    # Summary plot globale per la classe MITM
    plt.figure()
    shap.summary_plot(
        shap_values_mitm,
        X_multi_trans_sample,
        feature_names=feature_names_multi,
        show=False,
        max_display=20,
    )
    save_figure(FIGURES_DIR / "shap_summary_multiclass_mitm.png")

    # Selezione di un campione MITM dal test set
    y_test_labels_mitm = label_encoder.inverse_transform(y_test_m)
    mitm_candidates = np.where(y_test_labels_mitm == "mitm")[0]
    if len(mitm_candidates) > 0:
        sample_idx = int(mitm_candidates[0])
        x_single = best_multi_preprocessor.transform(X_test_m.iloc[[sample_idx]])
        if hasattr(x_single, "toarray"):
            x_single_for_shap = x_single.toarray()
        else:
            x_single_for_shap = np.asarray(x_single)

        if isinstance(shap_values, list):
            shap_single = shap.TreeExplainer(best_multi_model).shap_values(x_single_for_shap)[mitm_index][0]
            base_value = explainer.expected_value[mitm_index]
        else:
            shap_single = shap_values_mitm[0]
            base_value = expected_value_mitm

        plt.figure()
        shap.force_plot(
            base_value,
            shap_single,
            x_single_for_shap[0],
            feature_names=feature_names_multi,
            matplotlib=True,
            show=False,
        )
        save_figure(FIGURES_DIR / "shap_force_mitm.png")
    else:
        print("[WARN] Nessun campione MITM disponibile nel test set per il force plot.")

    shap_rank = np.abs(shap_values_mitm).mean(axis=0)
    shap_rank_df = pd.DataFrame({
        "feature": feature_names_multi,
        "mean_abs_shap": shap_rank,
    }).sort_values("mean_abs_shap", ascending=False)

    display(shap_rank_df.head(20))
    shap_rank_df.to_csv(OUTPUT_DIR / "shap_feature_importance_mitm.csv", index=False)
else:
    print("[WARN] SHAP non disponibile: sezione saltata.")


,feature,mean_abs_shap
3,num__src_bytes,0.571822
9,num__dst_ip_bytes,0.549415
6,num__src_pkts,0.335473
2,num__duration,0.286332
23,cat__service_http,0.256228
17,cat__proto_tcp,0.243779
0,num__src_port,0.226574
7,num__src_ip_bytes,0.224158
1,num__dst_port,0.168940
37,cat__conn_state_SF,0.155843


<Figure size 1000x600 with 0 Axes>

## 3.10 Validazione esterna su CIC-IoT-Dataset2023

La pipeline viene applicata al task binario sul secondo dataset per misurare la trasferibilità del framework e il cross-dataset generalization gap rispetto a TON_IoT.


In [25]:
# =========================================================================
# 3.10 VALIDAZIONE ESTERNA — FEATURE ENGINEERING UNIFICATA
# =========================================================================

from section_310_unified_feature_engineering import (
    build_unified_features_ton,
    build_unified_features_cic,
    build_unified_preprocessor,
    compute_distribution_shift,
    UNIFIED_NUMERIC_FEATURES
)

# ─────────────────────────────────────────────────────────────
# CONFIG
# ─────────────────────────────────────────────────────────────

import kagglehub
from pathlib import Path
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.base import clone
from sklearn.metrics import precision_recall_curve

assert "label" in df.columns, "TON dataset must contain 'label' column"

# ─────────────────────────────────────────────────────────────
# 1. CIC LOAD
# ─────────────────────────────────────────────────────────────

path = kagglehub.dataset_download("akashdogra/cic-iot-2023")
CIC_DATA_DIR = Path(path)

csv_files = list(CIC_DATA_DIR.rglob("*.csv"))
assert len(csv_files) > 0

cic_file = csv_files[0]

chunks = []
for chunk in pd.read_csv(
    cic_file,
    chunksize=100_000,
    engine="python",
    on_bad_lines="skip"
):
    if "label" not in chunk.columns:
        continue
    chunks.append(chunk.sample(frac=0.05, random_state=RANDOM_STATE))

cic_raw = pd.concat(chunks, ignore_index=True)
print(f"CIC loaded: {cic_raw.shape}")

# ─────────────────────────────────────────────────────────────
# 2. FEATURE ENGINEERING
# ─────────────────────────────────────────────────────────────

# =========================
# TON CLEAN
# =========================
ton_full = df.copy()
ton_full = ton_full.replace([np.inf, -np.inf], np.nan)
ton_full = ton_full.dropna(subset=["label"]).reset_index(drop=True)

X_ton_unified = build_unified_features_ton(ton_full)
y_ton = ton_full["label"].reset_index(drop=True)

# rimuove colonne completamente vuote
X_ton_unified = X_ton_unified.dropna(axis=1, how="all")

assert len(X_ton_unified) == len(y_ton), "Mismatch TON"

# =========================
# CIC CLEAN
# =========================
cic_raw = cic_raw.replace([np.inf, -np.inf], np.nan)
cic_raw = cic_raw.dropna(subset=["label"]).reset_index(drop=True)

X_cic_unified = build_unified_features_cic(cic_raw)

#  rimuove colonne vuote
X_cic_unified = X_cic_unified.dropna(axis=1, how="all")

# label realistico
y_cic_raw = cic_raw["label"].astype(str).str.lower()

attack_keywords = [
    "attack", "ddos", "dos", "bot", "injection",
    "bruteforce", "scan", "malicious"
]

y_cic = y_cic_raw.apply(
    lambda x: 1 if any(k in x for k in attack_keywords) else 0
).astype(int)

print("TON shape:", X_ton_unified.shape)
print("CIC shape:", X_cic_unified.shape)

# ─────────────────────────────────────────────────────────────
# 3. ALIGN FEATURE SPACE 
# ─────────────────────────────────────────────────────────────

common_features = [
    col for col in X_ton_unified.columns
    if col in X_cic_unified.columns
]

X_ton_unified = X_ton_unified[common_features].copy()
X_cic_unified = X_cic_unified[common_features].copy()

print(f"[ALIGN] common features: {len(common_features)}")

# safety check
assert X_ton_unified.shape[1] > 0, "No shared features TON/CIC"
assert X_cic_unified.shape[1] > 0, "No shared features TON/CIC"

# ─────────────────────────────────────────────────────────────
# 4. DOMAIN SHIFT
# ─────────────────────────────────────────────────────────────

shift_df = compute_distribution_shift(X_ton_unified, X_cic_unified)
display(shift_df)

shift_df.to_csv(
    OUTPUT_DIR / "unified_feature_domain_shift.csv",
    index=False
)

print("\n[DEBUG] CIC numeric means:")
print(X_cic_unified[UNIFIED_NUMERIC_FEATURES].mean(numeric_only=True))

# ─────────────────────────────────────────────────────────────
# 5. TRAIN
# ─────────────────────────────────────────────────────────────

X_train, X_test, y_train, y_test = train_test_split(
    X_ton_unified,
    y_ton,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y_ton
)

preprocessor = build_unified_preprocessor()
models = get_models(task="binary")

pipelines = {}
results_source = []

for name, model in models.items():

    model = clone(model)

    if hasattr(model, "class_weight"):
        model.set_params(class_weight="balanced")

    pipe = Pipeline([
        ("preprocessor", clone(preprocessor)),
        ("model", model)
    ])

    pipe.fit(X_train, y_train)
    pipelines[name] = pipe

    res, _, _ = evaluate_binary_pipeline(pipe, X_test, y_test, name)
    results_source.append(res)

source_df = pd.DataFrame(results_source)
display(source_df)

# ─────────────────────────────────────────────────────────────
# 6. THRESHOLD CALIBRATION
# ─────────────────────────────────────────────────────────────

def find_best_threshold(model, X_val, y_val):
    probs = model.predict_proba(X_val)[:, 1]
    p, r, thresholds = precision_recall_curve(y_val, probs)
    f1 = 2 * (p * r) / (p + r + 1e-9)
    return thresholds[np.argmax(f1)]

best_thresholds = {
    name: find_best_threshold(pipe, X_test, y_test)
    for name, pipe in pipelines.items()
}

# ─────────────────────────────────────────────────────────────
# 7. CROSS-DOMAIN TEST
# ─────────────────────────────────────────────────────────────

results_target = []

for name, pipe in pipelines.items():

    try:
        res, _, _ = evaluate_binary_pipeline(
            pipe,
            X_cic_unified,
            y_cic,
            name
        )

        threshold = best_thresholds.get(name, 0.5)

        probs = pipe.predict_proba(X_cic_unified)[:, 1]
        preds = (probs >= threshold).astype(int)

        res["threshold"] = threshold

        results_target.append(res)

    except Exception as e:
        print(f"[ERROR] {name}: {e}")

target_df = pd.DataFrame(results_target)
display(target_df)

# ─────────────────────────────────────────────────────────────
# 8. FINAL CHECK
# ─────────────────────────────────────────────────────────────

print("\n[CIC LABEL DISTRIBUTION]")
print(y_cic.value_counts(normalize=True))

print("\n[PREDICTION DISTRIBUTION]")
first_model = list(pipelines.values())[0]
preds = first_model.predict(X_cic_unified)

print(pd.Series(preds).value_counts(normalize=True))

CIC loaded: (11934, 47)
TON shape: (190474, 13)
CIC shape: (11934, 13)
[ALIGN] common features: 13


,feature,source_mean,target_mean,norm_shift
5,pkt_asymmetry,0.307224,3.646786,5.948750
8,flow_duration_sec,0.346167,2.575709,2.318009
6,payload_mean_fwd,1.532176,4.249366,1.345233
0,bytes_total,2.696308,6.580819,1.139730
1,bytes_src,1.992753,4.161724,0.822631
3,pkts_total,1.615949,2.348149,0.688161
2,bytes_dst,2.155785,4.333412,0.664702
7,payload_mean_bwd,1.725208,0.675432,0.406669
4,byte_asymmetry,-0.097844,0.018638,0.239881
9,flow_rate,4.338041,5.125985,0.139189



[DEBUG] CIC numeric means:
bytes_total          6.580819
bytes_src            4.161724
bytes_dst            4.333412
pkts_total           2.348149
byte_asymmetry       0.018638
pkt_asymmetry        3.646786
payload_mean_fwd     4.249366
payload_mean_bwd     0.675432
flow_duration_sec    2.575709
flow_rate            5.125985
dtype: float64
[LightGBM] [Info] Number of positive: 118747, number of negative: 33632
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.005812 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 2504
[LightGBM] [Info] Number of data points in the train set: 152379, number of used features: 10
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
[LightGBM] [Info] Start training from score 0.000000


,model,accuracy,precision,recall,f1,roc_auc,pr_auc,inference_ms_per_1k,model_size_mb
0,Logistic Regression,0.929938,0.958585,0.951191,0.954874,0.940802,0.963364,4.28641,0.005091
1,Random Forest,0.974931,0.969631,0.999124,0.984156,0.987093,0.993352,96.61443,27.546702
2,XGBoost,0.974170,0.969111,0.998686,0.983676,0.986996,0.993361,11.58460,0.851671
3,LightGBM,0.974459,0.970105,0.997979,0.983844,0.987342,0.993476,17.86577,1.330688


,model,accuracy,precision,recall,f1,roc_auc,pr_auc,inference_ms_per_1k,model_size_mb,threshold
0,Logistic Regression,0.886207,0.960023,0.913007,0.935925,0.654353,0.927733,14.40112,0.005091,0.401008
1,Random Forest,0.905815,0.911519,0.992912,0.950476,0.319968,0.884641,93.44297,27.546702,0.596667
2,XGBoost,0.909335,0.917670,0.989137,0.952065,0.614214,0.942764,5.26525,0.851671,0.616690
3,LightGBM,0.911681,0.948350,0.954985,0.951656,0.742821,0.955425,11.29221,1.330688,0.210763



[CIC LABEL DISTRIBUTION]
label
1    0.910256
0    0.089744
Name: proportion, dtype: float64

[PREDICTION DISTRIBUTION]
1    0.865678
0    0.134322
Name: proportion, dtype: float64


## 3.11 BASELINE DEEP LEARNING (MLP)
deep learning baseline for comparison

In [26]:
# =========================================================================
# 3.11 BASELINE DEEP LEARNING — MLP
# =========================================================================

from sklearn.neural_network import MLPClassifier
import time

mlp = MLPClassifier(
    hidden_layer_sizes=(64, 32),
    activation="relu",
    solver="adam",
    max_iter=30,
    random_state=RANDOM_STATE,
    early_stopping=True
)

mlp_pipe = Pipeline([
    ("preprocessor", clone(preprocessor)),
    ("model", mlp)
])

start = time.time()
mlp_pipe.fit(X_train, y_train)
train_time = time.time() - start

res_mlp, _, _ = evaluate_binary_pipeline(
    mlp_pipe,
    X_test,
    y_test,
    "MLP (DL baseline)"
)

res_mlp["train_time_sec"] = train_time

print(res_mlp)

{'model': 'MLP (DL baseline)', 'accuracy': 0.971019818873868, 'precision': 0.9671493478900403, 'recall': 0.9966652069929599, 'f1': 0.9816854678168547, 'roc_auc': 0.9843068156676971, 'pr_auc': 0.9925059060125683, 'inference_ms_per_1k': 5.0588599988259375, 'model_size_mb': 0.08133888244628906, 'train_time_sec': 10.585958242416382}


In [27]:
results_source.append(res_mlp)
source_df = pd.DataFrame(results_source)
display(source_df)

,model,accuracy,precision,recall,f1,roc_auc,pr_auc,inference_ms_per_1k,model_size_mb,train_time_sec
0,Logistic Regression,0.929938,0.958585,0.951191,0.954874,0.940802,0.963364,4.28641,0.005091,NaN
1,Random Forest,0.974931,0.969631,0.999124,0.984156,0.987093,0.993352,96.61443,27.546702,NaN
2,XGBoost,0.974170,0.969111,0.998686,0.983676,0.986996,0.993361,11.58460,0.851671,NaN
3,LightGBM,0.974459,0.970105,0.997979,0.983844,0.987342,0.993476,17.86577,1.330688,NaN
4,MLP (DL baseline),0.971020,0.967149,0.996665,0.981685,0.984307,0.992506,5.05886,0.081339,10.585958


## CROSS-DOMAIN TEST — MLP BASELINE

In [28]:
# =========================================================================
# CROSS-DOMAIN TEST — MLP BASELINE
# =========================================================================

from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.base import clone

results_mlp_cic = []

# ─────────────────────────────────────────────────────────────
# 1. DEFINIZIONE MLP (stesso setting del training implicito)
# ─────────────────────────────────────────────────────────────

mlp = MLPClassifier(
    hidden_layer_sizes=(64, 32),
    activation="relu",
    solver="adam",
    max_iter=50,
    random_state=RANDOM_STATE,
    early_stopping=True,
    n_iter_no_change=10,
    verbose=False
)

# Pipeline separata 
mlp_pipe = Pipeline([
    ("preprocessor", clone(preprocessor)),  # stesso feature engineering
    ("scaler", StandardScaler()),           # fondamentale per MLP
    ("model", mlp)
])

# ─────────────────────────────────────────────────────────────
# 2. TRAIN SU TON (stesso setup degli altri modelli)
# ─────────────────────────────────────────────────────────────

mlp_pipe.fit(X_train, y_train)

# ─────────────────────────────────────────────────────────────
# 3. EVALUATION SU CIC (CROSS-DOMAIN)
# ─────────────────────────────────────────────────────────────

res_mlp_cic, _, _ = evaluate_binary_pipeline(
    mlp_pipe,
    X_cic_unified,
    y_cic,
    "MLP (DL baseline)"
)

results_mlp_cic.append(res_mlp_cic)

mlp_cic_df = pd.DataFrame(results_mlp_cic)
display(mlp_cic_df)

# ─────────────────────────────────────────────────────────────
# 4. PREDICTION DISTRIBUTION CHECK
# ─────────────────────────────────────────────────────────────

mlp_preds_cic = mlp_pipe.predict(X_cic_unified)

print("\n[MLP CIC - Prediction distribution]")
print(pd.Series(mlp_preds_cic).value_counts(normalize=True))

,model,accuracy,precision,recall,f1,roc_auc,pr_auc,inference_ms_per_1k,model_size_mb
0,MLP (DL baseline),0.899698,0.958626,0.929946,0.944068,0.708345,0.959597,8.72913,0.082392



[MLP CIC - Prediction distribution]
1    0.883023
0    0.116977
Name: proportion, dtype: float64


## 3.12 Esportazione finale

La versione aggiornata del notebook include anche:
- la validazione k-fold sul task binario;
- l'analisi SHAP sul miglior modello multiclass;
- la validazione esterna su CIC-IoT-Dataset2023.


In [29]:
print("File prodotti:")
for folder in [FIGURES_DIR, TABLES_DIR, MODELS_DIR, OUTPUT_DIR]:
    print(f"\n{folder.name}:")
    for p in sorted(folder.glob("*")):
        print(" -", p.name)

File prodotti:

figures:
 - binary_confusion_matrix_best.png
 - binary_f1_comparison.png
 - binary_latency_comparison.png
 - binary_pr_auc_comparison.png
 - binary_pr_curve_best.png
 - binary_roc_auc_comparison.png
 - binary_roc_curve_best.png
 - binary_size_comparison.png
 - class_distribution_binary.png
 - class_distribution_multiclass.png
 - correlation_heatmap.png
 - missing_values_top20.png
 - multiclass_accuracy_comparison.png
 - multiclass_confusion_matrix_best.png
 - multiclass_latency_comparison.png
 - multiclass_macro_f1_comparison.png
 - multiclass_per_class_f1_best.png
 - multiclass_pr_auc_macro_comparison.png
 - multiclass_roc_auc_macro_comparison.png
 - multiclass_size_comparison.png
 - multiclass_weighted_f1_comparison.png
 - shap_force_mitm.png
 - shap_summary_multiclass_mitm.png
 - ton_binary_cv_f1_comparison.png
 - ton_binary_cv_pr_auc_comparison.png
 - ton_binary_cv_roc_auc_comparison.png

tables:

models:
 - binary_lightgbm.joblib
 - binary_logistic_regression.jobli